# Parametric Layout of a Penetrating Neural Probe (GDSII)

Generates a single-shank silicon neural probe layout with `gdstk`: tapered tip, a column of recording sites,
metal traces routed in side lanes, a fan-out region, and bond pads. Exports a real GDSII file plus an SVG and a preview image.

**How to use:** Runtime > Run all. Change the inputs in the "Inputs" cell and run again. Files are written to the Colab file panel
(`neural_probe.gds`, `neural_probe.svg`, `layout_preview.png`).

**Layers**

| Layer | Purpose |
|---|---|
| 1 | Probe outline (DRIE through-etch) |
| 2 | Metal (sites, traces, pads) |
| 3 | Passivation openings (sites, pads) |
| 4 | Labels |

All dimensions are in micrometers. Design-rule values below are placeholders: replace them with the rules of your target process.

In [ ]:
import subprocess, sys
try:
    import gdstk
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdstk"])
    import gdstk

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

## Inputs

In [ ]:
# Shank
SHANK_WIDTH = 60.0          # um
SHANK_LENGTH = 5000.0       # um, from tip to the start of the fan-out
TIP_LENGTH = 100.0          # um, tapered tip

# Recording sites
N_SITES = 8
SITE_DIAMETER = 15.0        # um, passivation opening diameter
SITE_METAL_MARGIN = 2.0     # um, metal extends beyond the opening on each side
FIRST_SITE_Y = 150.0        # um from the tip
SITE_PITCH = 50.0           # um

# Traces
TRACE_WIDTH = 2.0           # um
TRACE_SPACING = 2.0         # um

# Fan-out and head
FAN_LENGTH = 1500.0         # um
HEAD_LENGTH = 300.0         # um
HEAD_WIDTH = 1000.0         # um
PAD_SIZE = 80.0             # um
PAD_PITCH = 100.0           # um
PAD_OPENING_MARGIN = 5.0    # um, opening is smaller than the pad metal on each side

# Design rules (PLACEHOLDERS)
MIN_WIDTH = 2.0
MIN_SPACING = 2.0
MIN_EDGE_MARGIN = 3.0       # metal to probe edge

LAYER_OUTLINE, LAYER_METAL, LAYER_OPENING, LAYER_LABEL = 1, 2, 3, 4

## Build the layout

In [ ]:
lib = gdstk.Library(unit=1e-6, precision=1e-9)
cell = lib.new_cell("NEURAL_PROBE")

half_w = SHANK_WIDTH / 2.0
half_h = HEAD_WIDTH / 2.0
y_fan_end = SHANK_LENGTH + FAN_LENGTH
y_head_end = y_fan_end + HEAD_LENGTH

# --- outline ---
outline = gdstk.Polygon([
    (0, 0), (half_w, TIP_LENGTH), (half_w, SHANK_LENGTH), (half_h, y_fan_end), (half_h, y_head_end),
    (-half_h, y_head_end), (-half_h, y_fan_end), (-half_w, SHANK_LENGTH), (-half_w, TIP_LENGTH),
], layer=LAYER_OUTLINE)
cell.add(outline)

# --- lane assignment: lower sites get the outer lanes so jogs never cross other traces ---
site_y = [FIRST_SITE_Y + i * SITE_PITCH for i in range(N_SITES)]
metal_r = SITE_DIAMETER / 2.0 + SITE_METAL_MARGIN
lane_pitch = TRACE_WIDTH + TRACE_SPACING
lane_start = metal_r + TRACE_SPACING + TRACE_WIDTH / 2.0

sides = {-1: [], +1: []}
for i in range(N_SITES):
    sides[-1 if i % 2 == 0 else +1].append(i)

lane_x = {}
for s, idx in sides.items():
    n = len(idx)
    for rank, i in enumerate(idx):
        k = n - 1 - rank                      # lowest site -> outermost lane
        lane_x[i] = s * (lane_start + k * lane_pitch)

# --- pads: lanes sorted left to right map to pads left to right (no crossings) ---
order = sorted(range(N_SITES), key=lambda i: lane_x[i])
pad_x = {site: (k - (N_SITES - 1) / 2.0) * PAD_PITCH for k, site in enumerate(order)}
pad_y = y_fan_end + HEAD_LENGTH / 2.0

for i in range(N_SITES):
    # site metal + passivation opening
    cell.add(gdstk.ellipse((0, site_y[i]), metal_r, tolerance=0.01, layer=LAYER_METAL))
    cell.add(gdstk.ellipse((0, site_y[i]), SITE_DIAMETER / 2.0, tolerance=0.01, layer=LAYER_OPENING))
    # trace
    pts = [(0, site_y[i]), (lane_x[i], site_y[i]), (lane_x[i], SHANK_LENGTH),
           (pad_x[i], y_fan_end), (pad_x[i], pad_y)]
    cell.add(gdstk.FlexPath(pts, TRACE_WIDTH, layer=LAYER_METAL))
    # pad metal + opening + label
    cell.add(gdstk.rectangle((pad_x[i] - PAD_SIZE / 2, pad_y - PAD_SIZE / 2),
                             (pad_x[i] + PAD_SIZE / 2, pad_y + PAD_SIZE / 2), layer=LAYER_METAL))
    o = PAD_SIZE / 2 - PAD_OPENING_MARGIN
    cell.add(gdstk.rectangle((pad_x[i] - o, pad_y - o), (pad_x[i] + o, pad_y + o), layer=LAYER_OPENING))
    for t in gdstk.text(str(i + 1), 20, (pad_x[i] - 6, pad_y - PAD_SIZE / 2 - 30), layer=LAYER_LABEL):
        cell.add(t)

lib.write_gds("neural_probe.gds")
cell.write_svg("neural_probe.svg")
print("Wrote neural_probe.gds and neural_probe.svg")

## Design-rule checks (simple)

In [ ]:
outer_lane = max(abs(x) for x in lane_x.values())
edge_margin = half_w - (outer_lane + TRACE_WIDTH / 2.0)
head_needed = N_SITES * PAD_PITCH
site_pitch_ok = SITE_PITCH >= 2 * metal_r + MIN_SPACING

checks = {
    "trace width >= min width": TRACE_WIDTH >= MIN_WIDTH,
    "trace spacing >= min spacing": TRACE_SPACING >= MIN_SPACING,
    f"edge margin ({edge_margin:.1f} um) >= {MIN_EDGE_MARGIN} um": edge_margin >= MIN_EDGE_MARGIN,
    "site metal discs do not touch": site_pitch_ok,
    f"head width fits pads (needs {head_needed:.0f} um)": HEAD_WIDTH >= head_needed + PAD_SIZE,
    "pad spacing >= min spacing": PAD_PITCH - PAD_SIZE >= MIN_SPACING,
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

print()
print(f"Probe length : {y_head_end / 1000:.2f} mm")
print(f"Shank cross-section : {SHANK_WIDTH:.0f} um wide, shank length {SHANK_LENGTH / 1000:.2f} mm")
print(f"Site area : {np.pi * (SITE_DIAMETER / 2) ** 2:.0f} um^2 each, {N_SITES} sites")

## Preview

In [ ]:
COLORS = {LAYER_OUTLINE: ("#d9d9d9", 1.0), LAYER_METAL: ("#d4a017", 1.0), LAYER_OPENING: ("#1f77b4", 0.6), LAYER_LABEL: ("#333333", 1.0)}

def draw(ax, xlim, ylim, title):
    for layer in (LAYER_OUTLINE, LAYER_METAL, LAYER_OPENING, LAYER_LABEL):
        color, alpha = COLORS[layer]
        for p in cell.get_polygons(layer=layer, datatype=0):
            ax.add_patch(MplPolygon(p.points, closed=True, facecolor=color, edgecolor="none", alpha=alpha))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_title(title)
    ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")

fig, ax = plt.subplots(1, 3, figsize=(16, 8), gridspec_kw={"width_ratios": [1, 0.35, 1.6]})
tip_top = FIRST_SITE_Y + (N_SITES - 1) * SITE_PITCH + 100
draw(ax[0], (-half_w - 20, half_w + 20), (-20, tip_top), "Tip and recording sites")
draw(ax[1], (-half_h, half_h), (-100, y_head_end + 100), "Full probe")
draw(ax[2], (-half_h - 20, half_h + 20), (SHANK_LENGTH - 100, y_head_end + 50), "Fan-out and pads")
plt.tight_layout()
plt.savefig("layout_preview.png", dpi=200)
plt.show()

## Verify the exported GDS

In [ ]:
lib2 = gdstk.read_gds("neural_probe.gds")
top = lib2.top_level()[0]
print("Cell:", top.name)
print("Polygons:", len(top.polygons), " Paths:", len(top.paths))
bb = top.bounding_box()
print(f"Bounding box (um): x {bb[0][0]:.1f} to {bb[1][0]:.1f}, y {bb[0][1]:.1f} to {bb[1][1]:.1f}")

## Next steps for the paper
1. Replace the placeholder design rules with the rules of the fabrication process you describe in the process-flow section.
2. Open `neural_probe.gds` in a viewer (for example KLayout on a computer) and include a layout figure in the paper.
3. Feed the real site diameter, trace width and trace length from this layout into the impedance/noise notebook (trace resistance and parasitic capacitance).
4. Describe the process flow: each layer here maps to a lithography/etch step, with the DRIE/RIE step for the outline and the passivation opening etch.